# Fine-tune RoBERTa Base với Feature Extractor trên PHEME Dataset
Notebook này nạp trực tiếp dữ liệu huấn luyện và kiểm thử từ bộ dữ liệu PHEME, tiến hành tự chia tập train/val và dùng RoBERTa base đóng vai trò như feature extractor (đóng băng các lớp base, chỉ huấn luyện classifier head). Code hoàn toàn độc lập, không import từ code cũ.

In [33]:
# import pandas as pd
# from sklearn.model_selection import train_test_split

# # 1. Đọc dữ liệu (Tự động tách val từ train)
# train_path = '/kaggle/input/datasets/bodoict/pheme-mrcd/train.csv'
# test_path = '/kaggle/input/datasets/bodoict/pheme-mrcd/test.csv'

# df_train_full = pd.read_csv(train_path)
# df_test = pd.read_csv(test_path)

# # Chuyển nhãn true/false thành 0, 1 (0: True/Non-rumor, 1: Fake/Rumor)
# # Ép kiểu về str và in thường để tránh lỗi khi map label
# df_train_full['label_enc'] = df_train_full['label'].astype(str).str.lower().map({'true': 0, 'false': 1})
# df_test['label_enc'] = df_test['label'].astype(str).str.lower().map({'true': 0, 'false': 1})

# # Chia tập train thành train/val (80% train, 20% validation)
# df_train, df_val = train_test_split(df_train_full, test_size=0.2, random_state=42, stratify=df_train_full['label_enc'])

# print("Số lượng train set:", len(df_train))
# print("Số lượng validation set:", len(df_val))
# print("Số lượng test set (Germanwings):", len(df_test))

import pandas as pd

# 1. Đọc dữ liệu
train_path = '/kaggle/input/datasets/bodoict/pheme-mrcd/train.csv'
test_path = '/kaggle/input/datasets/bodoict/pheme-mrcd/test.csv'

df_train_full = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# Chuyển nhãn true/false thành 0, 1 (0: True/Non-rumor, 1: Fake/Rumor)
# Ép kiểu về str và in thường để tránh lỗi khi map label
df_train_full['label_enc'] = df_train_full['label'].astype(str).str.lower().map({'true': 0, 'false': 1})
df_test['label_enc'] = df_test['label'].astype(str).str.lower().map({'true': 0, 'false': 1})

# Dùng sự kiện E4 (Charlie Hebdo) làm tập Validation
df_val = df_train_full[df_train_full['event_name'] == 'charliehebdo'].copy()
df_train = df_train_full[df_train_full['event_name'] != 'charliehebdo'].copy()

print("Số lượng train set (E1, E2, E3):", len(df_train))
print("Số lượng validation set (E4 - Charlie Hebdo):", len(df_val))
print("Số lượng test set (Germanwings):", len(df_test))

Số lượng train set (E1, E2, E3): 3254
Số lượng validation set (E4 - Charlie Hebdo): 2079
Số lượng test set (Germanwings): 469


In [34]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# 2. Khởi tạo Tokenizer và xây dựng Dataset độc lập
model_name = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

class PhemeDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=128)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = PhemeDataset(df_train['text'], df_train['label_enc'])
val_dataset = PhemeDataset(df_val['text'], df_val['label_enc'])
test_dataset = PhemeDataset(df_test['text'], df_test['label_enc'])

In [35]:
# 3. Khởi tạo Model theo mô hình Feature Extractor (đóng băng roberta base)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

for param in model.roberta.parameters():
    param.requires_grad = False

print("Đã đóng băng base model. Chúng ta sẽ chỉ huấn luyện classification head.")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Đã đóng băng base model. Chúng ta sẽ chỉ huấn luyện classification head.


In [36]:
import evaluate
import numpy as np
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback

# 4. Cấu hình metric đánh giá và TrainingArguments
metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = metric_f1.compute(predictions=predictions, references=labels)
    acc = metric_acc.compute(predictions=predictions, references=labels)
    return {**f1, **acc}

training_args = TrainingArguments(
    output_dir='./results_roberta_pheme_feature_extractor',
    num_train_epochs=50, 
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    learning_rate=1e-3,
    weight_decay=0.01, # Cập nhật Weight decay
    optim="adamw_torch", # Chỉ định rõ thuật toán tối ưu là AdamW
    warmup_steps=500,
    logging_dir='./logs_pheme',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,           
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)] 
)

print("Starting training...")
trainer.train()

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,F1,Accuracy
1,1.340009,1.184283,0.000000,0.779702
2,1.295176,1.108728,0.000000,0.779702
3,1.195664,1.057856,0.618619,0.816739
4,1.077549,0.890499,0.626363,0.818663
5,1.024316,1.094303,0.616541,0.754690
6,1.014633,0.975605,0.639344,0.788360


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=306, training_loss=1.1543969843122694, metrics={'train_runtime': 85.4955, 'train_samples_per_second': 1903.023, 'train_steps_per_second': 29.826, 'total_flos': 461525568873120.0, 'train_loss': 1.1543969843122694, 'epoch': 6.0})

In [37]:
from sklearn.metrics import classification_report, confusion_matrix

# 5. Đánh giá trên tập test (Germanwings Plane Crash)
print("Đánh giá tổng quan trên test dataset:")
metrics = trainer.evaluate(test_dataset)
print(metrics)

# Lấy prediction cụ thể để in Confusion Matrix & Classification Report
print("\n--- Đang tạo báo cáo chi tiết ---")
predictions = trainer.predict(test_dataset)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

classes = ["True/Non-rumor (0)", "Fake/Rumor (1)"]

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=classes))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_true, y_pred)
print(cm)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Đánh giá tổng quan trên test dataset:


{'eval_loss': 1.243516445159912, 'eval_f1': 0.6503667481662592, 'eval_accuracy': 0.6950959488272921, 'eval_runtime': 0.8983, 'eval_samples_per_second': 522.099, 'eval_steps_per_second': 4.453, 'epoch': 6.0}

--- Đang tạo báo cáo chi tiết ---


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



Classification Report:
                    precision    recall  f1-score   support

True/Non-rumor (0)       0.65      0.84      0.73       231
    Fake/Rumor (1)       0.78      0.56      0.65       238

          accuracy                           0.70       469
         macro avg       0.71      0.70      0.69       469
      weighted avg       0.71      0.70      0.69       469


Confusion Matrix:
[[193  38]
 [105 133]]
